[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlatcl/multilogue/blob/main/notebooks/02_extract.ipynb)

# 02 · Extract — narrow passes, checkpoints, quote support

Address begins when we ask the model for structured points. Multilogue keeps extraction *generic*: a kind name, a prompt template, a sentinel for "nothing here", and an optional **quote support** gate so each phrase must be grounded in the speaker's own words.

**Why checkpoints?** Model stages run through `StageRunner`: append-only JSONL keyed by item, with a fingerprint of prompt + model. Reruns resume; fingerprint changes supersede; replay needs no API. That is engineering for reproducible dialogue analysis, not a modelling claim.

**Why FakeClient here?** Pedagogy and CI should not require a paid key. The fake model returns bank phrases that appear in the synthetic units so quote checks can fire.

In [ ]:
#@title Setup (run first)
# Free path: synthetic units + FakeClient (no API key).
# Optional: load("polis", ...) downloads public open data into a local cache.

import sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "matplotlib", "pandas", "numpy", "scikit-learn", "pyyaml",
        "git+https://github.com/mlatcl/multilogue.git",
    ])
else:
    try:
        import multilogue  # noqa: F401
    except ImportError:
        _here = pathlib.Path.cwd().resolve()
        for _cand in (_here, _here.parent, *_here.parents):
            if (_cand / "pyproject.toml").is_file() and (_cand / "multilogue").is_dir():
                if str(_cand) not in sys.path:
                    sys.path.insert(0, str(_cand))
                break

from IPython.display import display, HTML, Markdown
from multilogue.notebook import setup

FIG = setup(figures_dir="outputs/notebook_figures")
print("Figures →", FIG)


In [ ]:
from collections import defaultdict
from multilogue.access import load
from multilogue.dummy import FakeClient
from multilogue.extract import make_extract_work, phrase_table, yield_report
from multilogue.llm import StageRunner
from multilogue import viz

units = load("synthetic", n_docs_per_group=3, units_per_doc=3)["units"]
units_by_id = {u["unit_id"]: u for u in units}
client = FakeClient()
runner = StageRunner(FIG.parent / "ckpt_extract", FIG.parent / "logs_extract",
                     max_workers=2, sleep=lambda _: None)

extraction = {}
for kind in ("concern", "benefit"):
    work = make_extract_work(
        client, kind=kind, system="extract",
        user_for_unit=lambda u, k=kind: u["text"],
        model="fake-small", sentinel=f"NO_{kind.upper()}", quote_threshold=0.8,
    )
    extraction[kind] = runner.run(
        f"extract_{kind}", units, lambda u: u["unit_id"], work,
        f"extract|{kind}|fake-small|v1", quiet=True,
    )

phrases = phrase_table(extraction, units_by_id, kinds=["concern", "benefit"],
                       extra_unit_fields=["group"])
display(phrases.head())
by_group = defaultdict(list)
for u in units:
    by_group[u["group"]].append(u["unit_id"])
yd = yield_report(extraction, dict(by_group))
display(yd)
viz.yield_bars(yd, path=FIG / "02_yield.png")

## Quote support

A phrase without a recoverable quote is easy for a model to invent. The `quote_support` score asks how well the quoted span sits in the unit text. Thresholds are a design choice — high enough to catch fabrications, low enough to tolerate light paraphrase in the quote field.

In [ ]:
from multilogue.text import quote_support

demo_text = units[0]["text"]
span = " ".join(demo_text.split()[:8])
print("support(exact span):", round(quote_support(span, demo_text), 3))
print("support(noise):", round(quote_support("completely unrelated words", demo_text), 3))
if "verified" in phrases.columns:
    display(phrases.groupby(["group", "kind"]).verified.mean().rename("pct_verified"))